# Actividad 9. Clasificación con red neuronal

Se entrena una **red neuronal multicapa con embeddings** para estimar la
probabilidad de que un evento sea de alto impacto.

* Embeddings para las siete variables categóricas.
* Dos capas ocultas (64 y 32 neuronas) con ReLU y dropout.
* Salida sigmoide y `pos_weight` para compensar el desbalance.
* **Cinco semillas** promediadas y calibración de temperatura con validación.

El conjunto de prueba 2024–2025 no se usó para elegir nada: solo se consulta
al final.

## 1. Configuración y resultados guardados

In [1]:

import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados" / "ml"
GRAFICAS = RAIZ / "resultados" / "graficas" / "ml"
TABLAS = RAIZ / "resultados" / "tablas"
for carpeta in (RESULTADOS, GRAFICAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

PALETA = {
    "purpura": "#4C1D95",
    "purpura_claro": "#8B5CF6",
    "rosa": "#FA8A9A",
    "amarillo": "#FFB71B",
    "rojo": "#EE3E29",
    "coral": "#FF8A5F",
    "azul": "#1B4576",
    "gris": "#7A7191",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)


def guardar(fig, nombre):
    ruta = GRAFICAS / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Figura guardada: {ruta.relative_to(RAIZ)}")

metricas = json.loads((MODELOS / "metricas.json").read_text(encoding="utf-8"))
clf = metricas["clasificacion"]
nn = clf["red_neuronal"]

val = pd.read_csv(RESULTADOS / "pred_nn_val.csv")
test = pd.read_csv(RESULTADOS / "pred_nn_test.csv")
historial = pd.read_csv(RESULTADOS / "historial_nn_clasificacion.csv")
calibracion = pd.read_csv(RESULTADOS / "calibracion_nn.csv")

print(f"Semillas entrenadas: {historial['semilla'].nunique()}")
print(f"Temperatura de calibración: {clf['temperaturas']['red_neuronal']:.3f}")
print(f"Umbral elegido en validación: {clf['umbrales']['red_neuronal']:.2f}")
print(f"Eventos de prueba: {len(test):,}")


Semillas entrenadas: 5
Temperatura de calibración: 1.018
Umbral elegido en validación: 0.41
Eventos de prueba: 813


## 2. Curvas de entrenamiento

Cada semilla se detiene cuando la ROC-AUC de validación deja de mejorar. El
promedio de las cinco reduce la varianza típica de una red pequeña.

In [2]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
for semilla, grupo in historial.groupby("semilla"):
    ejes[0].plot(grupo["epoca"], grupo["perdida"], alpha=0.75, label=f"Semilla {semilla}")
    ejes[1].plot(grupo["epoca"], grupo["auc_validacion"], alpha=0.75, label=f"Semilla {semilla}")
ejes[0].set_title("Pérdida de entrenamiento")
ejes[0].set_xlabel("Época")
ejes[0].set_ylabel("BCE con peso positivo")
ejes[1].set_title("ROC-AUC de validación")
ejes[1].set_xlabel("Época")
ejes[1].set_ylabel("ROC-AUC")
ejes[1].legend(frameon=False, fontsize=8, ncol=2)
guardar(fig, "actividad9_entrenamiento.png")


Figura guardada: resultados\graficas\ml\actividad9_entrenamiento.png


## 3. Métricas de validación y prueba

In [3]:
claves = ["roc_auc", "pr_auc", "f1", "precision", "recall", "brier", "log_loss", "exactitud"]
tabla = pd.DataFrame(
    {particion: {clave: nn[particion][clave] for clave in claves} for particion in ("validacion", "prueba")}
).rename(columns={"validacion": "Validación 2023", "prueba": "Prueba 2024–2025"})
display(tabla.style.format("{:.4f}"))
tabla.to_csv(TABLAS / "actividad9_metricas_nn.csv")

fig, eje = plt.subplots(figsize=(8.5, 4))
x = np.arange(len(claves))
eje.bar(x - 0.19, tabla["Validación 2023"], width=0.38, color=PALETA["amarillo"], label="Validación 2023")
eje.bar(x + 0.19, tabla["Prueba 2024–2025"], width=0.38, color=PALETA["purpura"], label="Prueba 2024–2025")
eje.set_xticks(x)
eje.set_xticklabels(["ROC-AUC", "PR-AUC", "F1", "Precisión", "Recall", "Brier", "Log-loss", "Exactitud"], rotation=25, ha="right")
eje.set_ylim(0, 1)
eje.set_title("Red neuronal: métricas por partición")
eje.legend(frameon=False)
guardar(fig, "actividad9_metricas.png")


,Validación 2023,Prueba 2024–2025
roc_auc,0.8121,0.8197
pr_auc,0.6823,0.7653
f1,0.6724,0.7044
precision,0.5939,0.6328
recall,0.7748,0.7944
brier,0.1651,0.1678
log_loss,0.5030,0.5111
exactitud,0.7450,0.7368


Figura guardada: resultados\graficas\ml\actividad9_metricas.png


## 4. Curvas ROC y precisión-recall

La ROC-AUC mide la capacidad de ordenar; la PR-AUC es más exigente cuando la
clase positiva es minoritaria. Se muestran validación y prueba para detectar
cambios de distribución.

In [4]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.6))
for nombre, datos_parte, color in [
    ("Validación 2023", val, PALETA["amarillo"]),
    ("Prueba 2024–2025", test, PALETA["purpura"]),
]:
    fpr, tpr, _ = roc_curve(datos_parte["y"], datos_parte["nn"])
    ejes[0].plot(fpr, tpr, color=color, linewidth=2, label=f"{nombre} · AUC {roc_auc_score(datos_parte['y'], datos_parte['nn']):.3f}")
    precision, recall, _ = precision_recall_curve(datos_parte["y"], datos_parte["nn"])
    ejes[1].plot(recall, precision, color=color, linewidth=2, label=f"{nombre} · AP {average_precision_score(datos_parte['y'], datos_parte['nn']):.3f}")

ejes[0].plot([0, 1], [0, 1], linestyle="--", color=PALETA["gris"], linewidth=1)
ejes[0].set_xlabel("Tasa de falsos positivos")
ejes[0].set_ylabel("Tasa de verdaderos positivos")
ejes[0].set_title("Curva ROC")
ejes[0].legend(frameon=False, loc="lower right", fontsize=9)
ejes[1].set_xlabel("Recall")
ejes[1].set_ylabel("Precisión")
ejes[1].set_title("Curva precisión-recall")
ejes[1].legend(frameon=False, loc="lower left", fontsize=9)
guardar(fig, "actividad9_curvas.png")


Figura guardada: resultados\graficas\ml\actividad9_curvas.png


## 5. Calibración

Una probabilidad está calibrada si, entre los eventos a los que se asigna 30 %,
aproximadamente 30 % resultan de alto impacto. La temperatura se ajustó con
validación y se aplicó después a prueba.

In [5]:
fig, eje = plt.subplots(figsize=(6.2, 5))
eje.plot([0, 1], [0, 1], linestyle="--", color=PALETA["gris"], label="Calibración perfecta")
eje.plot(
    calibracion["probabilidad_media"],
    calibracion["frecuencia_observada"],
    marker="o",
    color=PALETA["purpura"],
    linewidth=2,
    label="Red neuronal",
)
for _, fila in calibracion.iterrows():
    eje.annotate(f"n={int(fila['n'])}", (fila["probabilidad_media"], fila["frecuencia_observada"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
eje.set_xlabel("Probabilidad estimada")
eje.set_ylabel("Frecuencia observada")
eje.set_title("Calibración en prueba 2024–2025")
eje.legend(frameon=False)
guardar(fig, "actividad9_calibracion.png")


Figura guardada: resultados\graficas\ml\actividad9_calibracion.png


## 6. Matriz de confusión con el umbral de validación

In [6]:
from sklearn.metrics import confusion_matrix

umbral = nn["prueba"]["umbral"]
matriz = confusion_matrix(test["y"], (test["nn"] >= umbral).astype(int), labels=[0, 1])
fig, eje = plt.subplots(figsize=(5.2, 4.4))
imagen = eje.imshow(matriz, cmap="Purples")
for i in range(2):
    for j in range(2):
        eje.text(j, i, f"{matriz[i, j]:,}", ha="center", va="center", fontsize=14, color="white" if matriz[i, j] > matriz.max() / 2 else PALETA["purpura"])
eje.set_xticks([0, 1])
eje.set_yticks([0, 1])
eje.set_xticklabels(["Predice ordinario", "Predice alto"])
eje.set_yticklabels(["Real ordinario", "Real alto"])
eje.set_title(f"Matriz de confusión · umbral {umbral:.2f}")
guardar(fig, "actividad9_matriz.png")

print(f"Verdaderos negativos: {matriz[0, 0]:,}")
print(f"Falsos positivos    : {matriz[0, 1]:,}")
print(f"Falsos negativos    : {matriz[1, 0]:,}")
print(f"Verdaderos positivos: {matriz[1, 1]:,}")


Figura guardada: resultados\graficas\ml\actividad9_matriz.png
Verdaderos negativos: 344
Falsos positivos    : 148
Falsos negativos    : 66
Verdaderos positivos: 255


## 7. Intervalos de confianza

Con 813 eventos de prueba, una métrica puntual no basta. Se reportan
intervalos bootstrap al 95 % (1,000 remuestreos).

In [7]:
prueba = nn["prueba"]
intervalos = pd.DataFrame(
    [
        {"metrica": "ROC-AUC", "estimacion": prueba["roc_auc"], "ic95_inferior": prueba["roc_auc_ic95"][0], "ic95_superior": prueba["roc_auc_ic95"][1]},
        {"metrica": "PR-AUC", "estimacion": prueba["pr_auc"], "ic95_inferior": prueba["pr_auc_ic95"][0], "ic95_superior": prueba["pr_auc_ic95"][1]},
        {"metrica": "F1", "estimacion": prueba["f1"], "ic95_inferior": prueba["f1_ic95"][0], "ic95_superior": prueba["f1_ic95"][1]},
    ]
)
display(intervalos.round(4))
intervalos.to_csv(TABLAS / "actividad9_intervalos.csv", index=False)


,metrica,estimacion,ic95_inferior,ic95_superior
0,ROC-AUC,0.8197,0.7888,0.8496
1,PR-AUC,0.7653,0.7171,0.8080
2,F1,0.7044,0.6548,0.7313


## Conclusión

La red neuronal alcanza una ROC-AUC de **0.820** y un F1 de **0.704** en el
conjunto de prueba, con calibración razonable. La siguiente actividad compara
estos resultados con XGBoost, usando exactamente la misma partición y las
mismas variables.